# Mission 2 — Complex deep-tabular first-screen (clean 59D)

질문: 같은 clean 59D와 preprocessing 계약에서 TabM / FT-Transformer / Tabular ResNet의 practical model/pipeline-level first-screen 결과는 어떻게 다른가?

**새 모델은 아직 competition data로 학습하지 않았습니다.** Stage 6의 두 MLP는 저장 결과만 재사용합니다. final model 선정이나 성능 개선을 미리 가정하지 않습니다.

고정: Training만 사용, 59D feature order, row/key/label, call_id split, seed 42, threshold 0.5, internal-train-only StandardScaler, Accuracy. Validation은 transform/evaluation/checkpoint 선택만 수행하고 gradient나 BatchNorm 통계 업데이트에 사용하지 않습니다.

입력은 현재 한 발화의 기존 59D뿐입니다. text/gender/symptom/order/odd-even/절대 startAt·endAt/call context를 입력하지 않습니다. duration을 60번째 feature로 추가하지 않습니다. overlap·duration·call 정보는 Checkpoint I의 진단에만 사용합니다.

모델마다 설정 1개, search/sweep/tuning/automatic retry/fallback 없음. **Run All은 실제 세 모델을 학습합니다.** 사용자가 E/F/G 셀을 실행하여 학습을 결정합니다.

## Checkpoint 0 — Google Drive / repo root / Python path

Colab GPU runtime을 선택하세요. 기존 mount 경로 `/content/drive/MyDrive/prelim`이 실제 저장소인지 확인하고 필요한 경우 아래 Path만 수정하세요.
repo의 기존 src helper와 manifest/metadata/clean chain/transfer identity, Stage 6 RUN_ROOT 전체가 필요합니다. 원본은 기존 공유 ZIP 바로가기를 재사용하며 개인 Drive에 다시 업로드하지 않습니다.

In [ ]:
from pathlib import Path
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO_ROOT = Path('/content/drive/MyDrive/prelim').resolve()
if not (REPO_ROOT / 'src/dcc_data/mission2_complex_tabular_screen.py').is_file():
    raise FileNotFoundError('새 src와 기존 helper를 포함한 실제 REPO_ROOT를 확인하세요.')
sys.path.insert(0, str(REPO_ROOT / 'src'))
STAGE6_ROOT = REPO_ROOT / 'data/processed/mission2_model_family_screen/20261005T113752Z_8ab60cd0'
print('REPO_ROOT:', REPO_ROOT)
print('STAGE6_ROOT:', STAGE6_ROOT)

## Checkpoint A — dependency / official package API / CUDA

공식 구현: [TabM](https://github.com/yandex-research/tabm), [RTDL revisiting models](https://github.com/yandex-research/rtdl-revisiting-models/tree/main/package). deprecated `rtdl`을 사용하지 않습니다.

package를 고정 설치하며 Colab의 기존 CUDA Torch build를 유지합니다. Torch가 `>=2.6,<3` 범위 밖이면 중단하고 환경을 확인하세요. TabM의 dependency인 rtdl-num-embeddings는 설치하지만 별도 embedding은 추가하지 않습니다. 기존 clean loader의 간접 dependency인 openSMILE도 기존 pin을 유지합니다.

로컬 smoke: Python 3.12.7 / Torch 2.6.0+cpu / TabM 0.0.3 / RTDL 0.0.2. 실제 Colab Python·Torch·CUDA는 아래에서 기록하며, GPU smoke/API 실패나 dependency 충돌은 중단 사유입니다.

In [ ]:
import os
import importlib.metadata
import torch
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
print('Python:', sys.version)
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
if not (2, 6) <= torch_version < (3, 0) or not torch.cuda.is_available():
    raise RuntimeError('PyTorch >=2.6,<3와 CUDA GPU runtime이 필요합니다. 자동 fallback은 없습니다.')

In [ ]:
%pip install 'tabm==0.0.3' 'rtdl-revisiting-models==0.0.2' 'rtdl-num-embeddings==0.0.12' 'opensmile==2.6.0'

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
from dcc_data import mission2_model_family_screen as screen
from dcc_data import mission2_complex_tabular_screen as complex_screen
ENVIRONMENT = complex_screen.check_environment(device='cuda')
display(ENVIRONMENT)

## Checkpoint B — existing clean 59D load / row alignment

기존 shared Training staging·transfer SHA-256 proof·strict clean-cache loader를 그대로 재사용합니다. feature 추출/재생성이나 Official Validation 로드는 없습니다. 공유 ZIP을 /content에 푸는 과정은 기존 source byte 확인용입니다.
loader가 source/row/key/label/call alignment·feature order·NaN/Inf·clean fingerprint를 확인하고 기존 fixed split/train-only scaler helper를 재사용합니다. 검사 실패 시 다음 셀로 넘어가지 마세요.

In [ ]:
SHARED_ARCHIVE_ROOT = REPO_ROOT / 'data/raw_original/대학부 데이터'
RAW_ROOT = Path('/content/mission2_shared_training_raw')
TRANSFER_IDENTITY_PATH = REPO_ROOT / 'tmp/mission2_egemaps_transfer_identity.json'
if not RAW_ROOT.resolve().is_relative_to(Path('/content')) or RAW_ROOT.resolve().is_relative_to(Path('/content/drive')):
    raise ValueError('RAW_ROOT는 Drive 밖의 /content 임시 경로여야 합니다.')
context = screen.load_shared_screen(
    REPO_ROOT, screen.shared_training_archive_paths(SHARED_ARCHIVE_ROOT),
    RAW_ROOT, TRANSFER_IDENTITY_PATH,
)
print('Clean fingerprint:', context.contract['provenance_fingerprint'])
print('Train:', context.prepared.X_train_scaled.shape)
print('Internal Validation:', context.prepared.X_evaluation_scaled.shape)

## Checkpoint C — fixed split / scaler / Stage 6 references

확인값: train 711,467 rows·23,360 calls / validation 177,891 rows·5,840 calls / overlap 0 / seed 42 / threshold 0.5 / scaler internal train only / Official Validation used False.

Stage 6과 source identity·row/key/label·split·feature order·scaler mean_/scale_/var_·fit scope를 확인하고 두 reference prediction을 exact-align합니다. **환경 간 scaled matrix의 SHA-256은 정보 기록이며 hard gate가 아닙니다.** 로컬 NumPy/sklearn 버전 차이로 발생한 byte hash 차이를 이유로 중단하지 않습니다. 기존 Stage 6 prediction을 추측하거나 refit하지 않습니다.

In [ ]:
references = complex_screen.load_stage6(context, STAGE6_ROOT)
display(context.contract['split'])
print('Scaler fit scope:', context.contract['scaler_fit_scope'])
print('Scaler fitted rows:', int(context.prepared.scaler.n_samples_seen_))
print('Official Validation used:', context.contract['official_validation_used'])
display(references['comparison'])
print('Scaled matrix hashes (informational):')
display({'current': {k: context.contract['identity'][k] for k in ('train_matrix', 'validation_matrix')},
         'stage6': references['saved_matrix_hashes']})

## Checkpoint D — frozen configs / one new RUN_ROOT

세 모델 모두 batch 256 / max epoch 100 / patience 10 / seed 42 / threshold 0.5. internal Validation Accuracy가 **엄격히 개선**된 epoch만 best로 저장하며 동점이면 이전 best를 유지합니다. Validation은 gradient·preprocessing fit·BatchNorm update에 들어가지 않습니다. scheduler 없음.

- TabM: 공식 no-embedding 기본형 3×512, k=32, dropout 0.1, AdamW lr=0.002 / wd=0.0003.
- FT-Transformer: 공식 3-block 기본형, width 192, 8 heads, attention dropout 0.2, FFN multiplier 4/3·dropout 0.1, residual dropout 0, 공식 optimizer groups, AdamW lr=0.0001 / wd=0.00001.
- Tabular ResNet: 공식 README 2-block 예제, width 192 / hidden multiplier 2 / dropout 0.15·0.0, BatchNorm, 공식 [end-to-end example](https://github.com/yandex-research/rtdl-revisiting-models/blob/main/package/example.ipynb)의 AdamW lr=3e-4 / wd=1e-5를 사용.

batch·epoch cap·patience는 메모리와 first-screen budget을 위한 공통 engineering 선택입니다. 논문 최적 설정 재현이나 pure architecture effect라고 표현하지 않습니다. Stage 6 sklearn MLP와 training recipe가 다르며 comparison metadata에도 남깁니다.

**이 셀은 한 번만 실행합니다. E–I에서 RUN_ROOT를 재생성하지 않습니다.** 기존 artifact나 같은 파일을 덮어쓰지 않습니다. 실패/OOM/NaN/Inf/reload 불일치 후 같은 run에서 다음 모델을 실행하면 중단합니다.

In [ ]:
FROZEN_CONFIGS = {name: complex_screen.get_config(name) for name in complex_screen.CANDIDATES}
display(FROZEN_CONFIGS)
if 'RUN_ROOT' in globals():
    raise RuntimeError('이미 RUN_ROOT가 있습니다. D를 반복 실행하지 말고 기존 run을 유지하세요.')
RUN_ROOT = complex_screen.create_run(REPO_ROOT, context, references, ENVIRONMENT)
print('RUN_ROOT:', RUN_ROOT)

## Checkpoint E — TabM

아래 셀을 실행하면 frozen config 1개로 실제 full-data 학습합니다. 학습·evaluation·best checkpoint 저장·reload 일치 검사는 src 공통 runner가 수행합니다. 오류가 발생하면 후속 fit을 계속하지 마세요.

In [ ]:
tabm_metrics = complex_screen.run_candidate('tabm', context, RUN_ROOT)
display(tabm_metrics)

## Checkpoint F — FT-Transformer

아래 셀을 실행하면 frozen config 1개로 실제 full-data 학습합니다. 학습·evaluation·best checkpoint 저장·reload 일치 검사는 src 공통 runner가 수행합니다. 오류가 발생하면 후속 fit을 계속하지 마세요.

In [ ]:
ft_transformer_metrics = complex_screen.run_candidate('ft_transformer', context, RUN_ROOT)
display(ft_transformer_metrics)

## Checkpoint G — Tabular ResNet

아래 셀을 실행하면 frozen config 1개로 실제 full-data 학습합니다. 학습·evaluation·best checkpoint 저장·reload 일치 검사는 src 공통 runner가 수행합니다. 오류가 발생하면 후속 fit을 계속하지 마세요.

In [ ]:
tabular_resnet_metrics = complex_screen.run_candidate('tabular_resnet', context, RUN_ROOT)
display(tabular_resnet_metrics)

## Checkpoint H — overall comparison

세 후보가 성공한 뒤 한 번 저장합니다. Stage 6 MLP reference·Stronger MLP를 함께 표시하며 두 기준에 대한 delta를 계산합니다. class Recall·FP/FN·balanced Accuracy·HCE·gap·시간을 함께 봅니다. HCE는 오답 중 confidence >=0.9의 개수입니다.

작은 Accuracy gain만으로 final model이나 안정적인 개선을 확정하지 않습니다. 현재 결과와 recipe/backend/time 차이를 먼저 읽은 뒤 I에서 분석할 후보를 정하세요.

In [ ]:
comparison = complex_screen.save_comparison(RUN_ROOT)
display(comparison)
for candidate in complex_screen.CANDIDATES:
    print(candidate, 'confusion:', complex_screen.screen._read_json(RUN_ROOT / candidate / 'metrics.json')['confusion_matrix'])
print('Saved:', RUN_ROOT / 'comparison.csv')

## Checkpoint I — selected paired / error slice analysis

전체 세 모델을 자동으로 깊게 분석하지 않습니다. 아래 list는 **TabM의 ensemble 구조를 확인하기 위한 기본 분석 대상**이며, H를 읽은 뒤 의미 있는 후보로 변경할 수 있습니다. 이 선택은 학습 config를 바꾸지 않습니다.

Stronger MLP 대비 both correct / both wrong / regression / correction / net correction을 확인합니다. 기본 분석 metadata는 기존 Stage 4 parquet의 record_key/call_id/y_true가 exact-align될 때만 사용합니다. 저장된 duration·overlap·두 weak slice는 기존 helper 정의를 재사용합니다. `TRACKING_PATH` 파일이 없으면 코드가 `tracking = None`으로 설정하여 overall/class/confidence/paired 분석만 수행하며 해당 slice를 만들었다고 보고하지 않습니다.

active/silence/low_information은 기존 명시적 Boolean mask가 exact-align된 경우만 지원합니다. 기본 Stage 4 parquet에는 이 mask가 없으므로 분석 metadata에 누락으로 기록합니다. 임의 cutoff를 만들지 않습니다. call별 error concentration은 진단용입니다.

In [ ]:
import pandas as pd
ANALYSIS_CANDIDATES = ['tabm']  # H를 읽은 뒤 실제 분석할 후보 이름만 선택
TRACKING_PATH = REPO_ROOT / 'data/processed/mission2_standard_representation_comparison/clean59d_egemaps88d_20261004T190153_4dbd7177/comparison/validation_predictions.parquet'
tracking = pd.read_parquet(TRACKING_PATH) if TRACKING_PATH.is_file() else None
paired, slices = complex_screen.save_paired_analysis(RUN_ROOT, ANALYSIS_CANDIDATES, tracking)
display(paired)
display(slices)
print('Saved:', RUN_ROOT / 'paired_comparison.csv')

## Artifact / execution handoff

정상 실행 순서: **0 → A(환경 확인·고정 설치·API 확인) → B → C → D 한 번 → E → F → G → H → I**. E/F/G를 실행할 때만 실제 모델을 학습합니다. package를 변경해야 하면 runtime을 restart하고 0부터 다시 시작하세요. 결과를 보고 config를 바꾸거나 실패 뒤 retry/fallback하지 않습니다.

`data/processed/mission2_complex_tabular_screen/<RUN_ID>/`:
- contract.json / environment.json / scaler.json
- stage6_reference/comparison.csv 및 두 reference predictions snapshot
- tabm/ · ft_transformer/ · tabular_resnet/: config.json / metrics.json / predictions.parquet / checkpoint.pt / history.csv
- comparison.csv / paired_comparison.csv / paired_slices.csv / error_concentration_by_call.csv / analysis_metadata.json

Stage 6 결과 root는 읽기만 합니다. 새 후보 prediction에는 record_key/call_id/y_true/P(speaker=1)/prediction/confidence/correct/error_type를 저장합니다. 실패 후보는 FAILED reason을 남기고 예외를 전파합니다. checkpoint reload는 같은 runtime에서 exact validation probability 일치를 요구합니다.

이 Notebook은 Colab full-data 실행을 위해 준비한 미실행 파일입니다. local CPU smoke와 actual CUDA/full-data 성능 검증은 별개입니다.